# Talk score – Colab runner

Runs the pipeline in `talk_score/` (see its README).
**Runtime → Change runtime type → T4 GPU** before starting (FinBERT needs a GPU).

All data is stored on Google Drive (`DATA_DIR`), so you can close the tab or lose the connection:
rerun the setup cells, then the cell you were on. Every stage continues where it stopped.

**Two phases**
* **Now:** firm list built from public data, industries = Fama-French 12 (from SEC SIC codes).
* **Later**, when the Datastream list with TRBC codes arrives: section 7. Everything already
  downloaded and scored is reused.

## 1. Setup (run every time you open the notebook)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/BMF5324'
DATA_DIR = f'{BASE}/talk_data'                 # all outputs go here
LM_DICT  = f'{BASE}/Loughran-McDonald_MasterDictionary_1993-2024.csv'   # check the exact file name
SEC_USER_AGENT = 'Your Name your.email@u.nus.edu'   # SEC requires name + email

UNIVERSE = f'{DATA_DIR}/universe.csv'          # phase 1: public list (built in step 2)
# UNIVERSE = f'{BASE}/sp500_universe.csv'      # phase 2: Datastream list (section 7)

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.environ['SEC_USER_AGENT'] = SEC_USER_AGENT

In [ ]:
# Get the code. If the repo is private: upload the talk_score folder to BASE on Drive and
# set CODE_DIR = f'{BASE}/talk_score' instead of cloning.
CODE_DIR = '/content/group-work/talk_score'
if not os.path.exists(CODE_DIR):
    !git clone -b claude/talk-score-calculation-nn01qa https://github.com/yannikoleschiffner-afk/group-work /content/group-work
%cd {CODE_DIR}
!pip -q install -r requirements.txt

## 2. Firm list from public data (only once)

All tickers in the S&P 500 at any year-end 2010–2024, including firms that left the index,
from the public history at github.com/fja05680/sp500. Company names and CIKs come from
Wikipedia's S&P 500 page.

In [ ]:
!python -m talk.cli --data-dir "$DATA_DIR" universe-github

## 3. Match firms to SEC CIKs

In [ ]:
!python -m talk.cli --data-dir "$DATA_DIR" firms --universe "$UNIVERSE"

In [ ]:
import pandas as pd
m = pd.read_csv(f'{DATA_DIR}/firm_cik_map.csv')
review = pd.read_csv(f'{DATA_DIR}/firms_review.csv')

# (a) Ticker renames found automatically (e.g. FB -> META): quick sanity check.
display(m[m.match_method.str.startswith('renamed_to')][['firm_id', 'match_method', 'last_year', 'cik']])

# (b) Firms without a match: look up the CIK at https://www.sec.gov/edgar/search/
#     and add "firm_id,cik" lines to DATA_DIR/firm_cik_overrides.csv, then rerun step 3.
#     cand1_* columns show the best guess; firms that were in the index only in 2010-2011
#     matter least.
print(len(review), 'firms to check')
pd.set_option('display.max_rows', 400)
review

In [ ]:
# Helper: create the overrides file with a header if it does not exist yet.
p = f'{DATA_DIR}/firm_cik_overrides.csv'
if not os.path.exists(p):
    open(p, 'w').write('firm_id,cik\n')
print(open(p).read())

## 4. Download 10-Ks (a few hours; rerun the cell if Colab disconnects)

In [ ]:
# Try a handful of firms first:
!python -m talk.cli --data-dir "$DATA_DIR" download --limit-firms 5

In [ ]:
!python -m talk.cli --data-dir "$DATA_DIR" download

## 5. Scoring

In [ ]:
!python -m talk.cli --data-dir "$DATA_DIR" score --method lm --lm-dict "$LM_DICT"
!python -m talk.cli --data-dir "$DATA_DIR" score --method finbert --batch-size 128

In [ ]:
# Check: how many FinBERT-environmental sentences does the keyword prefilter miss? (want >= 0.90)
!python -m talk.cli --data-dir "$DATA_DIR" score --method finbert --no-prefilter --sample 30
a = pd.read_csv(f'{DATA_DIR}/scores_finbert.csv').set_index('accession')
b = pd.read_csv(f'{DATA_DIR}/scores_finbert_noprefilter.csv').set_index('accession')
j = b.join(a, rsuffix='_pre', how='inner')
print('prefilter recall (environmental sentences):', round(j['fb_n_env_pre'].sum() / j['fb_n_env'].sum(), 3))
print('prefilter recall (positive environmental):  ', round(j['fb_n_env_pos_pre'].sum() / j['fb_n_env_pos'].sum(), 3))

## 6. Panel and percentiles (phase 1: Fama-French 12 industries)

In [ ]:
!python -m talk.cli --data-dir "$DATA_DIR" panel --universe "$UNIVERSE" --industry-ff12

In [ ]:
p = pd.read_csv(f'{DATA_DIR}/talk_panel.csv')
by_year = p.groupby('year').agg(
    firms=('firm_id', 'size'),
    with_talk=('fb_talk_raw', lambda s: s.notna().mean()),
    fulltext=('section_source', lambda s: (s == 'fulltext').mean()),
    item7_missing=('item7_short', 'mean'),
    talk_mean=('fb_talk_raw', 'mean'),
    env_share=('fb_env_share', 'mean'),
)
display(by_year.round(4))
print('corr FinBERT vs LM talk_raw:', round(p[['fb_talk_raw', 'lm_talk_raw']].corr().iloc[0, 1], 3))
display(p.groupby('industry')['fb_env_share'].mean().sort_values(ascending=False).round(4))

In [ ]:
# 20 random sentences labelled environmental + positive, to eyeball
s = pd.read_json(f'{DATA_DIR}/sentences_finbert.jsonl.gz', lines=True).drop_duplicates(['accession', 'sentence'])
pos = s[(s.fb_env) & (s.fb_tone == 'positive')]
for t in pos.sample(min(20, len(pos)), random_state=1).sentence:
    print('-', t[:300])

## 7. Later: switch to the Datastream list with TRBC codes

1. Put the Datastream file (`isin, name, ticker, year, trbc`, one row per firm and year) at
   `BASE/sp500_universe.csv`.
2. In the setup cell, switch `UNIVERSE` to that file and rerun the setup cells.
3. Run the cells below. Only firms the public list missed are downloaded and scored.

In [ ]:
!python -m talk.cli --data-dir "$DATA_DIR" firms --universe "$UNIVERSE"
# check firms_review.csv as in step 3, then:
!python -m talk.cli --data-dir "$DATA_DIR" download
!python -m talk.cli --data-dir "$DATA_DIR" score --method lm --lm-dict "$LM_DICT"
!python -m talk.cli --data-dir "$DATA_DIR" score --method finbert --batch-size 128
!python -m talk.cli --data-dir "$DATA_DIR" panel --universe "$UNIVERSE" --industry-col trbc